In [10]:
import torch, torch.nn as nn, torch.optim as optim, time

device = "cuda" if torch.cuda.is_available() else "cpu"

# Dummy CNN
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, 3, 1, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, 10)
    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

model = Net().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# random data
x = torch.randn(1024, 3, 64, 64, device=device)
y = torch.randint(0, 10, (1024,), device=device)

print("Running FP32 training...")
start = time.time()
for epoch in range(5):
    opt.zero_grad()
    out = model(x)
    loss = loss_fn(out, y)
    loss.backward()
    opt.step()
torch.cuda.synchronize()
print(f"FP32 time: {time.time() - start:.3f}s")


Running FP32 training...
FP32 time: 0.174s


In [ ]:
import torch, torch.nn as nn, torch.optim as optim, time

device = "cuda" if torch.cuda.is_available() else "cpu"

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, 3, 1, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, 10)
    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

model = Net().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

x = torch.randn(1024, 3, 64, 64, device=device)
y = torch.randint(0, 10, (1024,), device=device)

print("Running mixed precision training...")
start = time.time()
for epoch in range(5):
    opt.zero_grad()
    with torch.cuda.amp.autocast():
        out = model(x)
        loss = loss_fn(out, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
torch.cuda.synchronize()
print(f"Mixed precision time: {time.time() - start:.3f}s")


Running mixed precision training...
Mixed precision time: 0.099s


/tmp/ipykernel_6851/4251265379.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_6851/4251265379.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # FP16 forward/backward


In [13]:

x = torch.randn(1024, 3, 64, 64, device=device,dtype=torch.float8_e8m0fnu)


NotImplementedError: "normal_kernel_cuda" not implemented for 'Float8_e8m0fnu'